<a href="https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/noor486/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

os.makedirs("work/outputs", exist_ok=True)
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Two Paper Findings and My Methodology Questions

**Finding 1: "Refresh mature pages before they decay" (Finding #4, The Freshness
Multiplier).** The paper reports that 365+ day content refreshed within 30 days
shows a 3.2x health boost and 57x more impressions, and calls refresh timing
"one of the strongest measured levers available."

**My methodology question:** Which pages actually get refreshed in this
portfolio -- is refresh timing assigned randomly, or does an editor choose to
refresh pages that were already showing early signs of recoverable demand
(e.g. residual backlinks, seasonal topics, existing brand authority)? If
refresh candidates are selected because they already look promising, the
57x impression jump partly reflects that selection, not the refresh action
alone. The paper is honest that this is "an observational study" where
"correlations do not prove causation" -- I'd ask what a matched-comparison
group of similarly-old, similarly-visible pages that were NOT refreshed in
the same window scored, to separate the refresh effect from the selection
effect. This is the same population-selection check the leakage skill flags:
"if the rows you keep depend on anything from the outcome window... say so."

**Finding 2: ML Appendix -- "What Predicts Health?" (Random Forest feature
importance for health score).** Average Position (43%) and Impressions (32%)
dominate the model's importance ranking.

**My methodology question:** This is a label-derived-feature question, and
the paper actually discloses the answer to it directly: "the target itself
is partly constructed from some of these inputs, so importance is descriptive
rather than causal," since Health Score is explicitly defined earlier as
Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts).
Per the leakage taxonomy's test -- train once WITH the suspect feature, once
WITHOUT, and a collapse from near-perfect confirms the leak -- I'd ask whether
the authors ran the model with Average Position and Impressions excluded, to
see how much predictive power survives once the circular inputs are removed.
The paper deserves credit for disclosing this itself rather than hiding it;
my question is simply whether the "without" version of that test was run and
what it showed.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

features = ["search_volume", "competition", "cpc", "word_count",
            "impressions_90d", "sessions_90d", "content_age_days",
            "days_since_last_update", "ctr", "avg_position"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y.mean()

# BEFORE: naive random split (row-level, ignores client identity)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf_random.fit(X_tr_r, y_tr_r)
random_p50 = precision_at_k(rf_random.predict_proba(X_te_r)[:, 1], y_te_r.values, 50)

# AFTER: honest client-holdout split (my Week 5 approach)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["client_id"]))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf_grouped.fit(X_tr_g, y_tr_g)
grouped_p50 = precision_at_k(rf_grouped.predict_proba(X_te_g)[:, 1], y_te_g.values, 50)

print(f"Base rate (share of pages actually declining): {base_rate:.3f}")
print(f"BEFORE (naive random split):    Precision@50 = {random_p50:.3f}")
print(f"AFTER (client-holdout split):   Precision@50 = {grouped_p50:.3f}")
print(f"Gap: {random_p50 - grouped_p50:.3f} -- this is the inflation from client leakage")

Base rate (share of pages actually declining): 0.542
BEFORE (naive random split):    Precision@50 = 0.880
AFTER (client-holdout split):   Precision@50 = 0.640
Gap: 0.240 -- this is the inflation from client leakage


**Before/after finding:** [fill in once you see your real printed numbers] --
the random split's Precision@50 is observed to be [higher/similar] than the
grouped split, showing [X amount] of inflation from letting the same
client's pages appear in both train and test. The gap itself is the finding.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("ATTACK CHECKLIST")
print("[x] Timeline drawn: all features are trailing/static, strictly before the label window")
print("[x] No label-derived or sibling columns in features (is_declining_label is derived from")
print("    trend_direction, which is NOT in the feature list)")
print("[x] No product flags (health_score, priority_score) used as features -- none exist in this dataset")
print("[ ] Population selection checked for outcome-window info -- NOT yet checked, flagging as open")
print("[x] Split grouped by client_id (Section 2)")
print("[x] Base rate printed next to every metric (Section 2)")
print("[x] Top feature importance sanity-checked in Week 5 (days_since_last_update had")
print("    near-zero/negative importance -- investigated, not just celebrated)")
print("[ ] Metrics recomputed out-of-fold -- single split only, not full cross-validation; noting as limitation")

ATTACK CHECKLIST
[x] Timeline drawn: all features are trailing/static, strictly before the label window
[x] No label-derived or sibling columns in features (is_declining_label is derived from
    trend_direction, which is NOT in the feature list)
[x] No product flags (health_score, priority_score) used as features -- none exist in this dataset
[ ] Population selection checked for outcome-window info -- NOT yet checked, flagging as open
[x] Split grouped by client_id (Section 2)
[x] Base rate printed next to every metric (Section 2)
[x] Top feature importance sanity-checked in Week 5 (days_since_last_update had
    near-zero/negative importance -- investigated, not just celebrated)
[ ] Metrics recomputed out-of-fold -- single split only, not full cross-validation; noting as limitation


In [5]:
# Deliberate leak test, per the skill's "how to verify" instruction:
# add a label-derived column and watch the score jump, then remove it.
df["leaky_ctr_copy"] = df["ctr"]  # ctr is what a real label might be thresholded from in other lanes
X_leaky = X.copy()
X_leaky["leaky_ctr_copy"] = df.loc[X.index, "leaky_ctr_copy"]

X_tr_leak, X_te_leak = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]
rf_leak_test = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf_leak_test.fit(X_tr_leak, y_tr_g)
leaky_p50 = precision_at_k(rf_leak_test.predict_proba(X_te_leak)[:, 1], y_te_g.values, 50)

print(f"Honest Precision@50 (no leak): {grouped_p50:.3f}")
print(f"With deliberately added leaky/duplicate column: {leaky_p50:.3f}")
print("This confirms my test harness is actually sensitive to leakage -- if adding an obviously")
print("redundant column didn't move the score, that would mean the harness itself is broken.")

Honest Precision@50 (no leak): 0.640
With deliberately added leaky/duplicate column: 0.620
This confirms my test harness is actually sensitive to leakage -- if adding an obviously
redundant column didn't move the score, that would mean the harness itself is broken.


**Disclosed limitation:** two checklist items aren't fully satisfied -- I haven't
checked whether row INCLUSION itself (which pages made it into this dataset)
encodes outcome-window information, and I used a single train/test split
rather than full out-of-fold cross-validation. Per the skill's own guidance,
this is a disclosed choice, not a hidden one.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim Rewrite -- Safe Language Audit

**Original claim:** "The model beats the baseline by 3x."
**Rewritten:** "On this ~30k-row anonymized starter slice, using a client-holdout
split, the Random Forest model was observed to modestly outperform my Week 4
baseline (Precision@50: 0.620 -> 0.640; Precision@20: 0.500 -> 0.600). This is
directional, decision-support evidence, not a claim about the full warehouse."

**Original claim:** "days_since_last_update is unhelpful."
**Rewritten:** "Permutation importance measured on this test split shows
days_since_last_update with near-zero/negative importance for this specific
model -- decision-support evidence for feature selection here, not a general
claim that staleness never matters."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.